# Lab 6: Optimizers

At the end of Lab 5, a two-layer network was trained on XOR with `SGD` and `MSELoss`. Training worked, but it needed thousands of steps, and for some seeds it did not converge at all. This lab changes the two pieces of the training loop that do not depend on the network: the **loss** and the **optimizer**.

In this lab:

1. **Part 1:** replace the mean squared error by the binary cross-entropy, the commonly-used loss for binary classification, and see why it trains a sigmoid output faster.
2. **Part 2:** look at a landscape where plain gradient descent is slow, whatever the learning rate.
3. **Part 3:** add momentum to `SGD`.
4. **Part 4:** write `RMSProp`, which gives every parameter its own step size.
5. **Part 5:** combine the two into `Adam`.
6. **Part 6:** compare `SGD` and `Adam` on XOR.

The new loss and optimizers are written in `minitorch/` and checked with pytest, as in Lab 5. They use the `Linear`, `Sigmoid` and `Sequential` modules of Lab 5, so these should pass their tests first.

In [ ]:
import sys
sys.path.append('../')
import warnings
warnings.filterwarnings("ignore")

%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt

import minitorch
from minitorch.module import Parameter
from minitorch.layer import Linear, Sigmoid, Sequential
from minitorch.loss import MSELoss, BCELoss
from minitorch.optim import SGD, RMSProp, Adam
from tests import lab06_checks as checks

The data, the plotting function and the convergence test of Lab 5 are reused. `train` is the training loop of Lab 5, with one addition: it also records the **accuracy**, the fraction of the four points classified correctly (output above 0.5 for target 1, below 0.5 for target 0), at every epoch. Losses of different kinds cannot be compared with each other, but accuracies can.

In [ ]:
X_xor = np.array([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])
y_xor = np.array([[0.0], [1.0], [1.0], [0.0]])

SEED = 4


def train(model, loss_fn, optimizer, X, y, n_epochs):
    losses, accuracies = [], []
    for _ in range(n_epochs):
        optimizer.zero_grad()                  # reset the gradients left by the previous step
        pred = model(X)                        # forward pass
        loss = loss_fn(pred, y)                # loss
        model.backward(loss_fn.backward())     # backward pass, from the loss to the input
        optimizer.step()                       # update the parameters
        losses.append(loss)
        accuracies.append(np.mean((pred > 0.5) == (y > 0.5)))
    return losses, accuracies


def has_converged(model):
    pred = model(X_xor)
    return bool(np.all((pred > 0.5) == (y_xor > 0.5)))


def plot_soft_decision_region(predict, X, y, ax=None, title=None, margin=0.25):
    show = ax is None
    if show:
        _, ax = plt.subplots()
    pad = margin * (X.max(axis=0) - X.min(axis=0))
    xs = np.linspace(X[:, 0].min() - pad[0], X[:, 0].max() + pad[0], 200)
    ys = np.linspace(X[:, 1].min() - pad[1], X[:, 1].max() + pad[1], 200)
    gx, gy = np.meshgrid(xs, ys)
    grid = np.column_stack([gx.ravel(), gy.ravel()])
    out = np.asarray(predict(grid)).reshape(gx.shape)
    filled = ax.contourf(gx, gy, out, levels=np.linspace(0, 1, 21), cmap="coolwarm", vmin=0, vmax=1)
    if out.min() < 0.5 < out.max():
        ax.contour(gx, gy, out, levels=[0.5], colors="k")
    ax.scatter(X[:, 0], X[:, 1], c=y[:, 0], cmap="coolwarm", vmin=0, vmax=1, edgecolors="k", s=60)
    ax.set_xlabel("$x_1$")
    ax.set_ylabel("$x_2$")
    if title is not None:
        ax.set_title(title)
    plt.colorbar(filled, ax=ax, label="output")
    if show:
        plt.show()

## Part 1: Binary cross-entropy

The output layer of the network applies a logistic function $a = \sigma(z) = \frac{1}{1+e^{-z}}$. $a$ is a number between 0 and 1, and the target $y$ is 0 or 1. The output can be read as the probability $p=a$, according to the network, that the target is 1. The **binary cross-entropy** (BCE) measures how unlikely the targets are under these probabilities:

$$L = -\frac{1}{N}\sum_i^N \big(y_i \log p_i + (1 - y_i) \log (1 - p_i)\big).$$

For one example, only one of the two terms is non-zero: the loss is $-\log p$ if $y = 1$ and $-\log(1 - p)$ if $y = 0$. It is close to 0 when the output is close to the target, and it grows without bound when the output goes to the wrong end: predicting $p = 0.001$ for a target 1 costs $-\log 0.001 \approx 6.9$, where the squared error would cost at most 1.

The logarithm of 0 is $-\infty$, which happens when the sigmoid saturates in floating point. `BCELoss` therefore clips the prediction to $[\varepsilon, 1 - \varepsilon]$ before taking logarithms, with $\varepsilon = 10^{-7}$. `np.clip` does this.

### Exercise 1.1: `BCELoss`

Implement `BCELoss` in `minitorch/loss.py`. It works like `MSELoss`: `forward(pred, target)` returns a scalar and saves what `backward` needs, and `backward()` returns the gradient with respect to `pred`. Use the clipped prediction in both the loss and its gradient. Test it with `python -m pytest tests/test_loss.py -k bce -q`.

### Exercise 1.2: the gradient that reaches the sigmoid

During training, the gradient of the loss flows back through the output sigmoid to $z$, its input. Write `grad_wrt_z(loss_fn, z, target)`, which uses a `Sigmoid` module and the loss module `loss_fn`: it runs the forward pass from `z` to the loss, then the backward pass, and returns the gradient of the loss with respect to `z`.

In [ ]:
def grad_wrt_z(loss_fn, z, target):
    """Gradient of loss_fn(sigmoid(z), target) with respect to z, computed with the modules."""
    ...

checks.grad_wrt_z(grad_wrt_z)

The cell below uses `grad_wrt_z` on a single example with target 1, for values of $z$ from $-10$ to $10$. At $z = -10$ the output is $\sigma(-10) \approx 4.5 \times 10^{-5}$: the network is confidently wrong. The plot shows the size of the gradient on a logarithmic scale.

In [ ]:
zs = np.linspace(-10, 10, 201)
target = np.ones((1, 1))
for loss_fn in [MSELoss(), BCELoss()]:
    grads = [grad_wrt_z(loss_fn, np.array([[z]]), target)[0, 0] for z in zs]
    plt.semilogy(zs, np.abs(grads), label=type(loss_fn).__name__)
plt.xlabel("$z$ (target 1)")
plt.ylabel("$|\\partial L / \\partial z|$")
plt.legend()
plt.show()

**Question 1.** Derive $\partial L / \partial z$ for each loss, for one example, using $\sigma'(z) = a(1 - a)$. Use the result to explain the plot: what happens to the gradient of each loss when the output is confidently wrong?

### Exercise 1.3: BCE on XOR

Repeat Exercise 5.1 of Lab 5 with each loss: build `Sequential(Linear(2, 2), Sigmoid(), Linear(2, 1), Sigmoid())` after `np.random.seed(SEED)`, and train it on XOR with `SGD` (learning rate `1.0`) for `5000` epochs, once with `MSELoss` and once with `BCELoss`. Store the accuracies in `acc_mse` and `acc_bce`, and plot them against the epoch on the same axes.

In [ ]:
np.random.seed(SEED)
model_mse = ...
acc_mse = ...

np.random.seed(SEED)
model_bce = ...
acc_bce = ...

checks.xor_mse_vs_bce(model_mse, acc_mse, model_bce, acc_bce)

# plot acc_mse and acc_bce
...

One seed is not enough to compare two methods. `first_fit` returns the first epoch at which all four points are classified correctly, or `None` if this never happens. The cell below runs both losses on the seeds $0, \dots, 19$ and reports how many runs converge and the median number of epochs they need.

In [ ]:
def first_fit(accuracies):
    hits = np.flatnonzero(np.asarray(accuracies) == 1.0)
    return int(hits[0]) if hits.size else None


def run_xor(loss_fn, make_optimizer, n_hidden, seed, n_epochs):
    np.random.seed(seed)
    model = Sequential(Linear(2, n_hidden), Sigmoid(), Linear(n_hidden, 1), Sigmoid())
    losses, accuracies = train(model, loss_fn, make_optimizer(model.parameters()), X_xor, y_xor, n_epochs)
    return model, losses, accuracies


def summary(loss_fn, make_optimizer, n_hidden, n_seeds=20, n_epochs=5000):
    fits = []
    for seed in range(n_seeds):
        model, _, accuracies = run_xor(loss_fn, make_optimizer, n_hidden, seed, n_epochs)
        fits.append(first_fit(accuracies) if has_converged(model) else None)
    epochs = [f for f in fits if f is not None]
    rate = len(epochs) / n_seeds
    return rate, (float(np.median(epochs)) if epochs else None)


for loss_fn in [MSELoss(), BCELoss()]:
    rate, median = summary(loss_fn, lambda params: SGD(params, lr=1.0), n_hidden=2)
    print(f"{type(loss_fn).__name__:8s} converged: {rate:.0%}   median epochs to fit: {median}")

**Question 2.** What changes when the MSE is replaced by the BCE, and what does not? Relate your answer to Question 1.

From now on, every network is trained with `BCELoss`.

## Part 2: Where gradient descent is slow

In Lab 4, gradient descent was slow on a quadratic bowl that is much steeper in some directions than in others. The same bowl, in two dimensions, is used here to see what the optimizers of this lab change:

$$f(\mathbf{x}) = \lambda_1 x_1^2 + \lambda_2 x_2^2, \qquad \lambda_1 = 0.1, \quad \lambda_2 = 5.$$

Its gradient is $(2\lambda_1 x_1, 2\lambda_2 x_2)$. The point $\mathbf{x}$ is stored in a `Parameter`, so the optimizers of `minitorch.optim` can move it exactly as they move the weights of a network: at each step, `run_on_bowl` zeroes the gradient, writes the gradient of $f$ into `.grad`, and calls `step()`. It returns the trajectory, an array of shape `(n_steps + 1, 2)`. `plot_trajectories` draws several trajectories over the contour lines of $f$.

In [ ]:
EIGENVALUES = np.array([0.1, 5.0])
X0 = np.array([8.0, 4.0])
BOWL_STEPS = 60


def f_bowl(x):
    return np.sum(EIGENVALUES * x ** 2)


def run_on_bowl(make_optimizer, x0=X0, n_steps=BOWL_STEPS):
    point = Parameter(np.array(x0, dtype=float))
    optimizer = make_optimizer([point])
    trajectory = [point.data.copy()]
    for _ in range(n_steps):
        optimizer.zero_grad()
        point.grad += 2 * EIGENVALUES * point.data
        optimizer.step()
        trajectory.append(point.data.copy())
    return np.array(trajectory)


def plot_trajectories(trajectories, title=None):
    xs = np.linspace(-10, 10, 200)
    ys = np.linspace(-6, 6, 200)
    gx, gy = np.meshgrid(xs, ys)
    values = EIGENVALUES[0] * gx ** 2 + EIGENVALUES[1] * gy ** 2
    plt.figure(figsize=(9, 5))
    plt.contour(gx, gy, values, levels=np.geomspace(0.05, 200, 15), cmap="viridis", linewidths=0.7)
    for name, trajectory in trajectories.items():
        plt.plot(trajectory[:, 0], trajectory[:, 1], "o-", markersize=2.5, linewidth=1,
                 label=f"{name}  (f = {f_bowl(trajectory[-1]):.3g})")
    plt.plot(*X0, "s", color="k", label="start")
    plt.plot(0, 0, "*", color="gold", markeredgecolor="k", markersize=14, label="minimum")
    plt.xlim(-10, 10)
    plt.ylim(-6, 6)
    plt.xlabel("$x_1$")
    plt.ylabel("$x_2$")
    if title is not None:
        plt.title(title)
    plt.legend(loc="lower left", fontsize=8)
    plt.show()

### Exercise 2.1: plain SGD on the bowl

Run `SGD` on the bowl with the learning rates `0.05`, `0.19` and `0.21`, and draw the three trajectories in one figure. The final value of $f$ is shown in the legend.

In [ ]:
...

**Question 3.** Which direction limits the learning rate, and which direction limits the speed of convergence? Why can no single learning rate make plain gradient descent fast on this bowl?

## Part 3: Momentum

Plain gradient descent forgets the past: each step only depends on the current gradient. On the bowl, the gradients along $x_2$ change sign at every step and the gradients along $x_1$ always point the same way. **Momentum** keeps a running sum of the past gradients, the *velocity* $\mathbf{v}$, and moves along it:

$$\mathbf{v} \leftarrow \mu \mathbf{v} + \mathbf{g}, \qquad \mathbf{p} \leftarrow \mathbf{p} - \text{lr} \cdot \mathbf{v},$$

where $\mathbf{g}$ is the gradient of the parameter $\mathbf{p}$ and $\mu \in [0, 1)$ is the momentum coefficient. With $\mu = 0$ this is plain gradient descent. With a constant gradient, the velocity grows to $\mathbf{g} / (1 - \mu)$, so $\mu = 0.9$ takes steps up to 10 times larger along directions where the gradient keeps its sign, while gradients that alternate in sign cancel out.

Each parameter has its own velocity, an array of its shape that starts at zero. `SGD.__init__` creates them in `self.velocities`, in the order of `self.parameters`.

### Exercise 3.1: `SGD` with momentum

Implement the update above in `SGD.step` in `minitorch/optim.py`. The velocities must be updated in place, so that they are kept from one step to the next. Test it with `python -m pytest tests/test_optim.py -k sgd -q`: the tests of Lab 5 must still pass.

### Exercise 3.2: momentum on the bowl

Run `SGD` with learning rate `0.1` and the momentum coefficients `0`, `0.5` and `0.9` on the bowl, and draw the three trajectories in one figure.

In [ ]:
...

**Question 4.** Describe the effect of momentum along each direction. What goes wrong when $\mu$ is too large?

## Part 4: RMSProp

Momentum still uses one learning rate for every parameter. **RMSProp** divides the step of every entry of every parameter by a running estimate of the size of its gradient:

$$\mathbf{s} \leftarrow \alpha \mathbf{s} + (1 - \alpha)\, \mathbf{g}^2, \qquad \mathbf{p} \leftarrow \mathbf{p} - \text{lr} \cdot \frac{\mathbf{g}}{\sqrt{\mathbf{s}} + \varepsilon},$$

where the square, the square root and the division are elementwise, $\alpha$ is close to 1 (default `0.99`) and $\varepsilon$ (default `1e-8`) prevents a division by zero. $\mathbf{s}$ is a running average of the squared gradient, so $\sqrt{\mathbf{s}}$ is a typical size of the gradient: an entry whose gradient is large takes smaller steps, and an entry whose gradient is small takes larger ones.

### Exercise 4.1: `RMSProp`

Implement `RMSProp.step` in `minitorch/optim.py`. The running averages are created in `self.squares` by `__init__`. Test it with `python -m pytest tests/test_optim.py -k rmsprop -q`.

### Exercise 4.2: RMSProp on the bowl

Run `RMSProp` with learning rate `0.1` on the bowl, and draw its trajectory together with that of `SGD` with learning rate `0.19`. Then print the first step, `trajectory[1] - trajectory[0]`, of each.

In [ ]:
...

**Question 5.** Explain the first step of RMSProp from its update rule. Why does the shape of the bowl matter much less to RMSProp than to SGD? What does the learning rate of RMSProp control?

## Part 5: Adam

**Adam** combines the two ideas: a running average $\mathbf{m}$ of the gradient, as in momentum, divided by a running average of its size, as in RMSProp:

$$\mathbf{m} \leftarrow \beta_1 \mathbf{m} + (1 - \beta_1)\, \mathbf{g}, \qquad \mathbf{v} \leftarrow \beta_2 \mathbf{v} + (1 - \beta_2)\, \mathbf{g}^2,$$

$$\hat{\mathbf{m}} = \frac{\mathbf{m}}{1 - \beta_1^t}, \qquad \hat{\mathbf{v}} = \frac{\mathbf{v}}{1 - \beta_2^t}, \qquad \mathbf{p} \leftarrow \mathbf{p} - \text{lr} \cdot \frac{\hat{\mathbf{m}}}{\sqrt{\hat{\mathbf{v}}} + \varepsilon},$$

where $t = 1, 2, \dots$ counts the steps. The defaults are $\beta_1 = 0.9$, $\beta_2 = 0.999$, $\varepsilon = 10^{-8}$ and a learning rate of $0.001$.

$\mathbf{m}$ and $\mathbf{v}$ start at zero, so in the first steps they are smaller than the averages they estimate. The division by $1 - \beta^t$ is the **bias correction**: it compensates for this start at zero, and tends to 1 as $t$ grows.

### Exercise 5.1: `Adam`

Implement `Adam.step` in `minitorch/optim.py`. The running averages are created in `self.means` and `self.squares` by `__init__`, and the step counter `self.t` starts at 0: increase it at the beginning of every step. Test it with `python -m pytest tests/test_optim.py -k adam -q`.

### Exercise 5.2: the first step

The cell below takes one step of `Adam` with learning rate `0.5` on the bowl and prints it.

In [ ]:
traj_adam = run_on_bowl(lambda params: Adam(params, lr=0.5), n_steps=1)
print("first step of Adam:", traj_adam[1] - traj_adam[0])

**Question 6.** Compute by hand the first step of Adam, with and without the bias correction, with the default $\beta_1$ and $\beta_2$ (neglect $\varepsilon$). Does the result depend on the gradient? Why is the bias correction needed?

### Exercise 5.3: all the optimizers on the bowl

Draw in one figure the trajectories of `SGD` (learning rate `0.19`), `SGD` with momentum `0.9` (learning rate `0.1`), `RMSProp` (learning rate `0.1`) and `Adam` (learning rate `0.5`) on the bowl.

In [ ]:
...

**Question 7.** Compare the four trajectories. Which features of momentum and of RMSProp do you recognise in the trajectory of Adam?

## Part 6: SGD against Adam on XOR

The bowl is a landscape chosen to make gradient descent slow. This part compares `SGD` and `Adam` on the task of Lab 5, the XOR network trained with `BCELoss`, using `run_xor` and `summary` from Part 1.

The two learning rates are not on the same scale. The step of `SGD` is the learning rate times the gradient, so a good learning rate depends on the size of the gradients. The step of `Adam` has a size of about its learning rate, whatever the size of the gradients, so its learning rate is the distance a parameter moves in one step.

### Exercise 6.1: one run

With 4 hidden neurons and the seed `SEED`, train one network with `SGD` (learning rate `1.0`) and one with `Adam` (learning rate `0.05`), for `2000` epochs each, using `run_xor`. Plot the two loss curves on the same axes (with a logarithmic scale for the loss), and the output of each trained network with `plot_soft_decision_region`.

In [ ]:
model_sgd, losses_sgd, acc_sgd = ...
model_adam, losses_adam, acc_adam = ...

checks.xor_sgd_vs_adam(model_sgd, losses_sgd, model_adam, losses_adam)

# plot the loss curves and the output of both networks
plt.semilogy(losses_sgd, label="SGD, lr=1.0")
plt.semilogy(losses_adam, label="Adam, lr=0.05")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.legend()
plt.show()

_, axes = plt.subplots(1, 2, figsize=(10, 4))
plot_soft_decision_region(model_sgd, X_xor, y_xor, ax=axes[0], title="SGD", margin=1.0)
plot_soft_decision_region(model_adam, X_xor, y_xor, ax=axes[1], title="Adam", margin=1.0)
plt.tight_layout()
plt.show()

### Exercise 6.2: over many seeds

Use `summary` (with `n_epochs=2000`) to compare `SGD` (learning rate `1.0`) and `Adam` (learning rate `0.05`) for 2, 3 and 4 hidden neurons: for each, the fraction of the 20 seeds that converge and the median number of epochs to fit. Present the results as a table. This cell trains 120 networks and takes a little while.

In [ ]:
...

**Question 8.** Is Adam faster? Does it converge more often? Use `run_xor` and `plot_soft_decision_region` on a seed where Adam fails with 2 hidden neurons to see what the network has learned, and explain why a better optimizer cannot fix it. What helps instead?

In [ ]:
...

### Exercise 6.3: sensitivity to the learning rate

A learning rate has to be chosen for every new problem, so it matters which learning rates work for each optimizer. For each of the learning rates in `LRS`, train networks with 4 hidden neurons for `2000` epochs on the seeds $0, \dots, 9$, with `SGD`, `SGD+momentum`, `RMSProp`, and with `Adam`, and compute the mean final loss. Plot the mean final loss against the learning rate for all optimizers, with logarithmic axes.

In [ ]:
LRS = [1e-4, 3e-4, 1e-3, 3e-3, 1e-2, 3e-2, 1e-1, 3e-1, 1.0, 3.0, 1e1, 3e1, 1e2, 3e2]

In [ ]:
def mean_final_loss(make_optimizer, n_seeds=10):
    # Your code
    ...

final_sgd = [mean_final_loss(lambda params: SGD(params, lr=lr)) for lr in LRS]
final_mom = [mean_final_loss(lambda params: SGD(params, lr=lr, momentum=0.5)) for lr in LRS]
final_rms = [mean_final_loss(lambda params: RMSProp(params, lr=lr)) for lr in LRS]
final_adam = [mean_final_loss(lambda params: Adam(params, lr=lr, betas=(0.9, 0.99))) for lr in LRS]

plt.loglog(LRS, final_sgd, "o-", label="SGD")
plt.loglog(LRS, final_mom, "o-", label="SGD+momentum")
plt.loglog(LRS, final_rms, "o-", label="RMSProp")
plt.loglog(LRS, final_adam, "o-", label="Adam")
plt.xlabel("learning rate")
plt.ylabel("mean final loss")
plt.legend()
plt.show()

**Question 9.** For each optimizer, which learning rates work? Why are the good learning rates of Adam so much smaller than those of SGD, and what goes wrong with Adam when its learning rate is too large? (Think about how far a weight moves in one step of Adam, compared with the size of the weights.) How about increasing the training steps to 5,000? Any changes does it bring?

## Key takeaways

- The binary cross-entropy is the loss for a sigmoid output: its gradient with respect to $z$ is $a - y$, so it does not vanish when the network is confidently wrong.
- Plain gradient descent uses one step size for every direction, which the steepest direction limits.
- Momentum accumulates the gradients that keep their sign and cancels those that oscillate.
- RMSProp, and Adam after it, give every parameter its own step size, of about the learning rate.
- A better optimizer reaches a minimum faster. It does not choose a better minimum: when training gets stuck because of the network or its initialization, changing the optimizer does not help.

> **Note.** Adam with a learning rate around $10^{-3}$ is the default choice in most of deep learning, and `torch.optim.Adam` has the same arguments as the `Adam` written here.

## Running the tests

Run all the tests of the loss and the optimizers together from a terminal, at the root of the project:

```
python -m pytest tests/test_loss.py tests/test_optim.py -q
```